```
SAM 2 (Foundation Model):
  → Pretrained on 11M images + 1B masks by Meta
  → You provide PROMPTS (clicks, boxes) instead of labels
  → Works on ANY object without training
  → Zero-shot: segment things it's never been trained to segment
  → Inference only (seconds, not hours)
```

```
SAM 1 (2023): Images only
  → Click once → get mask for that frame

SAM 2 (2024): Images AND video
  → Click once on frame 0 → mask propagates to ALL frames
  → Uses a memory bank to track objects across time
  → Can be corrected: click on frame 5 to fix a wrong mask

This is why SAM 2 matters: it enables video object segmentation
without ANY training data.
```

```
SAM 2   (July 2024):     https://dl.fbaipublicfiles.com/segment_anything_2/072824/
SAM 2.1 (Sept 2024):    https://dl.fbaipublicfiles.com/segment_anything_2/092824/

SAM 2.1 is better — improved mask quality and video tracking.
The original guide uses the older July URL → use September (2.1) instead.

Model sizes (all produce good results, vary in speed/accuracy):
  tiny   (~38MB)  → fastest, for rapid testing
  small  (~46MB)  → best for free Colab ← we use this
  base+  (~80MB)  → better accuracy
  large  (~224MB) → best accuracy, slow on T4
```

In [ ]:
import importlib
import os
import sys

BASE_DIR = '/content' if os.path.exists('/content') else os.getcwd()
SAM2_DIR = os.path.join(BASE_DIR, 'sam2')

if not os.path.exists(SAM2_DIR):
    print("Cloning SAM 2...")
    os.system(f"git clone https://github.com/facebookresearch/sam2.git {SAM2_DIR}")
    print("Installing...")
    os.system(f"cd {SAM2_DIR} && pip install -e . --quiet")
    print("SAM 2 installed")
else:
    print("Already cloned")

os.system("pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 --quiet")
os.system("pip install roboflow opencv-python matplotlib supervision --quiet")

os.chdir(BASE_DIR)
print("Working dir:", os.getcwd())

if SAM2_DIR not in sys.path:
    sys.path.insert(0, SAM2_DIR)

sam2 = importlib.import_module('sam2')
print(f"sam2 imported successfully from {sam2.__file__}")

In [ ]:
CHECKPOINT_DIR = os.path.join(BASE_DIR, 'sam2_checkpoints')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

CHECKPOINT_NAME = 'sam2.1_hiera_small.pt'
CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, CHECKPOINT_NAME)
CHECKPOINT_URL  = f'https://dl.fbaipublicfiles.com/segment_anything_2/092824/{CHECKPOINT_NAME}'

CONFIG_FILE = os.path.join(SAM2_DIR, 'configs', 'sam2.1', 'sam2.1_hiera_s.yaml')

if not os.path.exists(CHECKPOINT_PATH):
  os.system(f"wget -q -O {CHECKPOINT_PATH} {CHECKPOINT_URL}")

  if os.path.exists(CHECKPOINT_PATH):
    size_mb = os.path.getsize(CHECKPOINT_PATH) / 1e6
    print(f"Downloaded: {size_mb:.1f} MB")
  else:
    print(f"Error: Failed to download checkpoint from {CHECKPOINT_URL}")
else:
  size_mb = os.path.getsize(CHECKPOINT_PATH) / 1e6
  print(f"Checkpoint already exists: {size_mb:.1f} MB")

In [ ]:
if '/content/sam2' not in sys.path:
    sys.path.insert(0, '/content/sam2')

In [ ]:
import os
import sys
import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import warnings
warnings.filterwarnings('ignore')

from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

In [ ]:
CONFIG_FILE = 'configs/sam2.1/sam2.1_hiera_s.yaml'

sam2_model = build_sam2(
  config_file=CONFIG_FILE,
  ckpt_path=CHECKPOINT_PATH,
  device=device
)

predictor = SAM2ImagePredictor(sam2_model)
total_params = sum(p.numel() for p in sam2_model.parameters())
print(f"   Total parameters: {total_params/1e6:.1f}M")

In [ ]:
img_enc = sam2_model.image_encoder
print(f"1. Image Encoder  : {type(img_enc).__name__}")

if hasattr(img_enc, 'trunk'):
  print(f"   Backbone type : {type(img_enc.trunk).__name__}")
  print(f"   (Hiera = Hierarchical Vision Transformer — processes image once)")

if hasattr(sam2_model, 'sam_mask_decoder'):
  mask_dec = sam2_model.sam_mask_decoder
  print(f"\n2. Mask Decoder   : {type(mask_dec).__name__}")
  print(f"   Num mask tokens: {mask_dec.num_mask_tokens}  (3 masks + IoU token)")

if hasattr(sam2_model, 'sam_prompt_encoder'):
  prompt_enc = sam2_model.sam_prompt_encoder
  print(f"\n3. Prompt Encoder : {type(prompt_enc).__name__}")
  print(f"   Embed dim     : {prompt_enc.embed_dim}")

if hasattr(sam2_model, 'memory_encoder'):
    print(f"\n4. Memory Encoder : {type(sam2_model.memory_encoder).__name__}")
    print(f"   (New in SAM2 — enables video tracking across frames)")

if hasattr(sam2_model, 'memory_attention'):
    print(f"5. Memory Attention: {type(sam2_model.memory_attention).__name__}")
    print(f"   (Cross-attends current frame to memory bank for temporal consistency)")

In [ ]:
from roboflow import Roboflow
import glob
import os

API_KEY = 'YOUR_API_KEY_HERE'
WORKSPACE = 'YOUR_WORKSPACE_HERE'
PROJECT_NAME = 'YOUR_PROJECT_NAME_HERE'
VERSION = 1

try:
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)

    project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
    dataset = project.version(ROBOFLOW_VERSION).download("png-mask-semantic")
    dataset_images = glob.glob(f"{dataset.location}/train/*.jpg") + glob.glob(f"{dataset.location}/train/*.png")
    if dataset_images:
        ROBOFLOW_IMAGE_PATH = dataset_images[0]
        print(f"Dataset downloaded to: {dataset.location}")
        print(f"Selected image: {ROBOFLOW_IMAGE_PATH}")
    else:
        print("No images found in the dataset folder.")
except Exception as e:
    print(f"An error occurred during Roboflow setup: {e}")

In [ ]:
IMAGE_PATH = ROBOFLOW_IMAGE_PATH

image = cv2.imread(IMAGE_PATH)
assert image is not None, f"Failed to read image: {IMAGE_PATH}"
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
H, W = image.shape[:2]
print(f"Image dimensions: width={W}, height={H}")

fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(image)
ax.set_title(f'Roboflow Dataset Image: {os.path.basename(IMAGE_PATH)}')
plt.axis('on')
plt.show()

In [ ]:
IMAGE_PATH = ROBOFLOW_IMAGE_PATH

image = cv2.imread(IMAGE_PATH)
if image is None:
    raise FileNotFoundError(f"Could not find or open the Roboflow image at {IMAGE_PATH}")

image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
H, W = image.shape[:2]
print(f"Image dimensions: width={W}, height={H}")

fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(image)
ax.set_title('Hover to read (x, y) coordinates from the axis labels\n'
             'x = horizontal (left→right),  y = vertical (top→bottom)', fontsize=11)
ax.set_xlabel('x coordinate (horizontal)')
ax.set_ylabel('y coordinate (vertical)')
ax.plot(W//2, H//2, 'r+', markersize=20, markeredgewidth=2,
        label=f'Center ({W//2}, {H//2})')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print(f"\nImage size  : width={W}, height={H}")
print(f"Image center: x={W//2}, y={H//2}")

In [ ]:
with torch.inference_mode():
  predictor.set_image(image)

def show_mask(mask, ax, color=None, alpha=0.5):
  if color is None:
    color = np.array([1.0, 0.0, 0.0])
  h, w = mask.shape
  overlay = np.zeros((h, w, 4), dtype=float)
  overlay[mask] = np.array([*color, alpha])
  ax.imshow(overlay)

In [ ]:
def show_points(coords, labels, ax, size=200):
  pos = coords[labels == 1]
  neg = coords[labels == 0]

  ax.scatter(pos[:, 0], pos[:, 1], color='lime',   s=size, marker='*', zorder=10, label='FG point (1)')
  ax.scatter(neg[:, 0], neg[:, 1], color='red',    s=size, marker='X', zorder=10, label='BG point (0)')

In [ ]:
def show_box(box, ax, color='green', linewidth=2):
  x0, y0, x1, y1 = box
  rect = patches.Rectangle((x0, y0), x1-x0, y1-y0,linewidth=linewidth, edgecolor=color, facecolor='none')
  ax.add_patch(rect)

# **Single foreground point**

In [ ]:
POINT_X = W // 2
POINT_Y = H // 3

input_point = np.array([[POINT_X, POINT_Y]])
input_label = np.array([1])

with torch.inference_mode():
  masks, scores, logits = predictor.predict(
      point_coords=input_point,
      point_labels=input_label,
      multimask_output=True
  )

print(f" Output shapes:")
print(f"   masks  : {masks.shape}   (3 masks, H, W)")
print(f"   scores : {scores}   (confidence per mask)")
print(f"   logits : {logits.shape}  (raw logits, use for mask prompt)")

In [ ]:
best_idx = int(np.argmax(scores))
best_mask = masks[best_idx] > 0.0

fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(image)
show_mask(best_mask, ax, color=np.array([0, 0.8, 0]))
show_points(input_point, input_label, ax)
ax.set_title(
    f'Single Point Prompt  |  Score: {scores[best_idx]:.3f}\n'
    f'Mask covers {best_mask.sum():,} pixels ({best_mask.mean()*100:.1f}% of image)',
    fontsize=11
)
ax.legend(fontsize=9)
ax.axis('off')
plt.tight_layout(); plt.show()

# **Foreground + Background points (most powerful technique)**

In [ ]:
fg_points = np.array([[POINT_X, POINT_Y]])
bg_points = np.array([[POINT_X + W//4, POINT_Y]])

all_points = np.vstack([fg_points, bg_points])
all_labels = np.array([1, 0])

with torch.inference_mode():
    masks_refined, scores_refined, logits_refined = predictor.predict(
        point_coords    = all_points,
        point_labels    = all_labels,
        multimask_output = False
    )
refined_mask_bool = masks_refined[0] > 0.0

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

axes[0].imshow(image)
show_mask(best_mask, axes[0], color=np.array([0, 0.8, 0]))
show_points(input_point, input_label, axes[0])
axes[0].set_title(f'Single FG Point\nScore: {scores[best_idx]:.3f}', fontsize=11)
axes[0].axis('off')

axes[1].imshow(image)
show_mask(refined_mask_bool, axes[1], color=np.array([0, 0.5, 1]))
show_points(all_points, all_labels, axes[1])
axes[1].set_title(f'FG + BG Points (refined)\nScore: {scores_refined[0]:.3f}', fontsize=11)
axes[1].legend(fontsize=9)
axes[1].axis('off')

plt.suptitle('Point Prompt Comparison: Single vs FG+BG', fontsize=13)
plt.tight_layout(); plt.show()

# **Bounding Box Prompt**

In [ ]:
x_min, y_min, x_max, y_max = int(W * 0.2), int(H * 0.1), int(W * 0.8), int(H * 0.9)
input_box = np.array([x_min, y_min, x_max, y_max])

with torch.inference_mode():
    masks_box, scores_box, _ = predictor.predict(
        box=input_box,
        multimask_output=False
    )

mask_box_bool = masks_box[0] > 0.0

fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(image)
show_mask(mask_box_bool, ax, color=np.array([1, 0.5, 0]))
show_box(input_box, ax, color='yellow', linewidth=3)
ax.set_title(f'Box Prompt | Score: {scores_box[0]:.3f}')
ax.axis('off')
plt.show()

In [ ]:
with torch.inference_mode():
    masks_combo, scores_combo, _ = predictor.predict(
        box             = input_box,
        point_coords    = np.array([[POINT_X, POINT_Y]]),
        point_labels    = np.array([1]),
        multimask_output = False
    )

mask_box_bool = masks_box[0] > 0.0
mask_combo_bool = masks_combo[0] > 0.0

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

axes[0].imshow(image)
show_mask(mask_box_bool, axes[0], color=np.array([1, 0.5, 0]))
show_box(input_box, axes[0], color='yellow')
axes[0].set_title(f'Box only\nScore: {scores_box[0]:.3f}', fontsize=11)
axes[0].axis('off')

axes[1].imshow(image)
show_mask(mask_combo_bool, axes[1], color=np.array([0, 0.8, 0.5]))
show_box(input_box, axes[1], color='yellow')
show_points(np.array([[POINT_X, POINT_Y]]), np.array([1]), axes[1])
axes[1].set_title(f'Box + FG Point\nScore: {scores_combo[0]:.3f}', fontsize=11)
axes[1].axis('off')

plt.suptitle('Box Prompt vs Box + Point Combined', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
with torch.inference_mode():
    masks_r1, scores_r1, logits_r1 = predictor.predict(
        point_coords    = np.array([[POINT_X, POINT_Y]]),
        point_labels    = np.array([1]),
        multimask_output = False
    )
    masks_r2, scores_r2, logits_r2 = predictor.predict(
        point_coords    = np.array([[POINT_X, POINT_Y]]),
        point_labels    = np.array([1]),
        mask_input      = logits_r1,
        multimask_output = False
    )

mask_r1_bool = masks_r1[0] > 0.0
mask_r2_bool = masks_r2[0] > 0.0

print(f"Round 1 score: {scores_r1[0]:.4f}")
print(f"Round 2 score: {scores_r2[0]:.4f}  (should be >= round 1)")

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

axes[0].imshow(image)
show_mask(mask_r1_bool, axes[0], color=np.array([0.8, 0.2, 0]))
show_points(np.array([[POINT_X, POINT_Y]]), np.array([1]), axes[0])
axes[0].set_title(f'Round 1\nScore: {scores_r1[0]:.4f}', fontsize=11)
axes[0].axis('off')

axes[1].imshow(image)
show_mask(mask_r2_bool, axes[1], color=np.array([0.0, 0.8, 0.2]))
show_points(np.array([[POINT_X, POINT_Y]]), np.array([1]), axes[1])
axes[1].set_title(f'Round 2 (with mask_input)\nScore: {scores_r2[0]:.4f}', fontsize=11)
axes[1].axis('off')

plt.suptitle('Mask Prompt Chaining: Iterative Refinement', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
with torch.inference_mode():
    masks_multi, scores_multi, _ = predictor.predict(
        point_coords=np.array([[POINT_X, POINT_Y]]),
        point_labels=np.array([1]),
        multimask_output=True
    )

labels_text = ['Whole Object', 'Object Part', 'Sub-part']
best_idx = np.argmax(scores_multi)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for i, (ax, label) in enumerate(zip(axes, labels_text)):
    ax.imshow(image)
    mask_bool = masks_multi[i] > 0.0
    color = np.array([0, 1, 0]) if i == best_idx else np.array([1, 0, 0])
    show_mask(mask_bool, ax, color=color)
    show_points(np.array([[POINT_X, POINT_Y]]), np.array([1]), ax)
    title_prefix = "BEST ★ " if i == best_idx else ""
    ax.set_title(f"{title_prefix}{label}\nScore: {scores_multi[i]:.3f}")
    ax.axis('off')

plt.tight_layout(); plt.show()

In [ ]:
mask_generator = SAM2AutomaticMaskGenerator(
    model=sam2_model,
    points_per_side=32,
    points_per_batch=64,
    pred_iou_thresh=0.7,
    stability_score_thresh=0.92,
    crop_n_layers=1,
    crop_n_points_downscale_factor=2,
)

with torch.inference_mode():
    auto_masks = mask_generator.generate(image)

print(f"Generated {len(auto_masks)} masks using optimized parameters.")

In [ ]:
def visualize_auto_masks(image, masks, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(12, 10))

    ax.imshow(image)
    # Filter out low-stability masks for cleaner visualization
    valid_masks = [m for m in masks if m['stability_score'] > 0.85]
    sorted_masks = sorted(valid_masks, key=lambda x: x['area'], reverse=True)

    H_img, W_img = image.shape[:2]
    overlay = np.zeros((H_img, W_img, 4), dtype=float)
    np.random.seed(42)

    for mask_data in sorted_masks:
        seg   = mask_data['segmentation']
        color = np.random.random(3)
        overlay[seg] = np.array([*color, 0.4]) # Slightly lower alpha for better visibility

    ax.imshow(overlay)

    # Only label the top 15 most significant masks to avoid clutter
    for i, mask_data in enumerate(sorted_masks[:15]):
        seg = mask_data['segmentation']
        cy, cx = np.where(seg)
        if len(cy) > 0:
            ax.text(cx.mean(), cy.mean(), str(i+1),
                    color='white', fontsize=8, ha='center', va='center',
                    fontweight='bold',
                    bbox=dict(facecolor='black', alpha=0.5, edgecolor='none', pad=1))
    return ax

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
axes[0].imshow(image); axes[0].set_title('Original'); axes[0].axis('off')
visualize_auto_masks(image, auto_masks, ax=axes[1])
axes[1].set_title(f'Refined Automatic Segmentation ({len(auto_masks)} masks)', fontsize=12)
axes[1].axis('off')
plt.tight_layout(); plt.show()

# **Video Segmentation**


> with own video



In [ ]:
from sam2.sam2_video_predictor import SAM2VideoPredictor

VIDEO_FILE_PATH = '/content/AQMwDt7tOGegz9pcVllKoW-N6hsw3o1Ezg5yP2NofTvi8leG1TLIkklEhZe6KEMjJe-nZ14t_RexSxmKomcM8SdQl45rZ5VjBt5xM-q4LIUezw.mp4'
VIDEO_FRAMES_DIR = '/content/video_frames'
os.makedirs(VIDEO_FRAMES_DIR, exist_ok=True)

if VIDEO_FILE_PATH is not None and os.path.exists(VIDEO_FILE_PATH):
  os.system(f"ffmpeg -i '{VIDEO_FILE_PATH}' -q:v 2 -vf 'fps=5' "
              f"{VIDEO_FRAMES_DIR}/%04d.jpg -y -loglevel quiet")
  frame_files = sorted([
      f for f in os.listdir(VIDEO_FRAMES_DIR)
      if f.endswith('.jpg')
  ])
else:
  num_frames = 8
  for i in range(num_frames):
    factor = 1.0 + (i - num_frames // 2) * 0.04
    frame_bgr = np.clip(image.astype(float) * factor, 0, 255).astype(np.uint8)
    cv2.imwrite(os.path.join(VIDEO_FRAMES_DIR, f"{i:04d}.jpg"), frame_bgr)
  frame_files = sorted(os.listdir(VIDEO_FRAMES_DIR))
  print(f" Created {num_frames} simulated frames in {VIDEO_FRAMES_DIR}")

print(f"   Frame list: {frame_files[:5]}{'...' if len(frame_files) > 5 else ''}")

In [ ]:
video_predictor = SAM2VideoPredictor.from_pretrained(
    model_id="facebook/sam2.1-hiera-small"
)
video_predictor = video_predictor.to(device)

In [ ]:
import sys
import os
import cv2

if 'decord' not in sys.modules:
  !pip install decord --quiet

with torch.inference_mode():
  inference_state = video_predictor.init_state(video_path=VIDEO_FRAMES_DIR)

  _, out_obj_ids, out_mask_logits = video_predictor.add_new_points_or_box(
      inference_state = inference_state,
      frame_idx       = 0,
      obj_id          = 1,
      points          = np.array([[POINT_X, POINT_Y]], dtype=np.float32),
      labels          = np.array([1], dtype=np.int32)
  )

  frame_results = {}

  for frame_idx, obj_ids, mask_logits in video_predictor.propagate_in_video(inference_state):
    mask = (mask_logits[0, 0] > 0.0).cpu().numpy()
    frame_results[frame_idx] = mask

  print(f" Propagated to {len(frame_results)} frames")

In [ ]:
n_show = min(len(frame_results), 8)
fig, axes = plt.subplots(1, n_show, figsize=(3.5 * n_show, 5))
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    frame_path = os.path.join(VIDEO_FRAMES_DIR, f'{i:04d}.jpg')
    if not os.path.exists(frame_path):
        ax.axis('off')
        continue
    frame = cv2.cvtColor(cv2.imread(frame_path), cv2.COLOR_BGR2RGB)
    ax.imshow(frame)
    if i in frame_results:
        show_mask(frame_results[i], ax, color=np.array([0, 0.8, 0.5]))
        coverage = frame_results[i].mean() * 100
        ax.set_title(f'Frame {i}\n{coverage:.1f}% coverage', fontsize=9)
    if i == 0:
        show_points(np.array([[POINT_X, POINT_Y]]), np.array([1]), ax)
        ax.set_xlabel('← annotated here', fontsize=8)
    ax.axis('off')

plt.suptitle('SAM 2 Video Tracking: Click Once on Frame 0 → Propagates Automatically', fontsize=11)
plt.tight_layout()
plt.show()

### Enhanced Video Segmentation Visualization
If the previous summary was too small, this cell creates a larger grid specifically for the video tracking results stored in `frame_results`.

In [ ]:
import matplotlib.pyplot as plt
import cv2
import os
import numpy as np

all_frame_indices = sorted(frame_results.keys())
if not all_frame_indices:
    print("Error: No segmentation results found in frame_results dictionary.")
else:
    step = max(1, len(all_frame_indices) // 8)
    selected_indices = all_frame_indices[::step][:8]

    n_cols = 4
    n_rows = (len(selected_indices) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 5 * n_rows))
    axes = axes.flatten()

    print(f"Visualizing segmentation for {len(selected_indices)} frames...")

    for i, idx in enumerate(selected_indices):
        frame_path = os.path.join(VIDEO_FRAMES_DIR, f'{(idx+1):04d}.jpg')

        if os.path.exists(frame_path):
            frame = cv2.imread(frame_path)
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            axes[i].imshow(frame)

            mask = frame_results[idx]

            if not np.any(mask):
                axes[i].set_title(f"Frame {idx}: EMPTY MASK", color='red')
            else:
                show_mask(mask, axes[i], color=np.array([0, 1, 0]), alpha=0.6)

                contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                for cnt in contours:
                    axes[i].plot(cnt[:, 0, 0], cnt[:, 0, 1], color='lime', linewidth=2)

                coverage = mask.mean() * 100
                axes[i].set_title(f"Frame {idx} ({coverage:.2f}% coverage)")
        else:
            axes[i].text(0.5, 0.5, f"File Not Found:\n{idx+1}.jpg", ha='center', color='red')

        axes[i].axis('off')

    for j in range(i + 1, len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    plt.show()



> with Roboflow datasets



In [ ]:
API_KEY = 'YOUR_API_KEY_HERE'
WORKSPACE = 'YOUR_WORKSPACE_HERE'
PROJECT_NAME = 'YOUR_PROJECT_NAME_HERE'
VERSION = 1

rf = Roboflow(api_key=RF_API_KEY)
project = rf.workspace(RF_WORKSPACE).project(RF_PROJECT)
dataset = project.version(RF_VERSION).download("sam2")

RF_DATASET_DIR = dataset.location
for split in ['train', 'valid', 'test']:
  split_dir = os.path.join(RF_DATASET_DIR, split)
  if os.path.exists(split_dir):
    imgs = [f for f in os.listdir(split_dir) if f.endswith(('.jpg', '.jpeg', '.png'))]
    print(f"   {split:6s}: {len(imgs)} images")

In [ ]:
RF_VIDEO_DIR = '/content/rf_video_frames'
os.makedirs(RF_VIDEO_DIR, exist_ok=True)

for split in ['train', 'valid', 'test']:
  split_dir = os.path.join(RF_DATASET_DIR, split)
  if os.path.isdir(split_dir):
    rf_images = sorted([
        f for f in os.listdir(split_dir)
        if f.endswith(('.jpg', '.jpeg', '.png'))
    ])
    if rf_images:
      break

MAX_FRAME = 10
rf_images = rf_images[:MAX_FRAME]

for idx, fname in enumerate(rf_images):
  src = os.path.join(split_dir, fname)
  dst = os.path.join(RF_VIDEO_DIR, f"{idx:04d}.jpg")
  img = cv2.imread(src)

  if img is not None:
    cv2.imwrite(dst, img)

first_frame_path = os.path.join(RF_VIDEO_DIR, '0000.jpg')
first_frame = cv2.imread(first_frame_path)
if first_frame is not None:
    first_frame = cv2.cvtColor(first_frame, cv2.COLOR_BGR2RGB)
    RF_H, RF_W = first_frame.shape[:2]
else:
    print(f"Error: Could not load {first_frame_path}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(first_frame)
ax.set_title(f'Roboflow Frame 0 ({RF_W}×{RF_H})\n'
             f'Adjust RF_CLICK_X, RF_CLICK_Y to click on your object of interest', fontsize=11)
ax.set_xlabel('x coordinate')
ax.set_ylabel('y coordinate')
ax.plot(RF_W//2, RF_H//2, 'r+', markersize=20, markeredgewidth=2,
        label=f'Center ({RF_W//2}, {RF_H//2})')
ax.legend()
plt.tight_layout()
plt.show()

print(f" {len(rf_images)} Roboflow frames ready in {RF_VIDEO_DIR}")
print(f"   Frame size: {RF_W}×{RF_H}")

### Roboflow Video Dataset Tracking
Now we will apply the SAM 2 video predictor to the frames we extracted from the Roboflow dataset.

In [ ]:
with torch.inference_mode():
    rf_inference_state = video_predictor.init_state(video_path=RF_VIDEO_DIR)
    _, rf_obj_ids, rf_mask_logits = video_predictor.add_new_points_or_box(
        inference_state = rf_inference_state,
        frame_idx       = 0,
        obj_id          = 1,
        points          = np.array([[RF_W//2, RF_H//2]], dtype=np.float32),
        labels          = np.array([1], dtype=np.int32)
    )

    rf_video_results = {}
    for frame_idx, obj_ids, mask_logits in video_predictor.propagate_in_video(rf_inference_state):
        mask = (mask_logits[0, 0] > 0.0).cpu().numpy()
        rf_video_results[frame_idx] = mask

    print(f"Successfully tracked object across {len(rf_video_results)} Roboflow frames.")

In [ ]:
n_show = len(rf_video_results)
fig, axes = plt.subplots(1, n_show, figsize=(4 * n_show, 5))
if n_show == 1: axes = [axes]

for i, ax in enumerate(axes):
    frame_path = os.path.join(RF_VIDEO_DIR, f'{i:04d}.jpg')
    if os.path.exists(frame_path):
        frame = cv2.cvtColor(cv2.imread(frame_path), cv2.COLOR_BGR2RGB)
        ax.imshow(frame)

        mask = rf_video_results[i]
        if np.any(mask):
            show_mask(mask, ax, color=np.array([1, 0.5, 0]), alpha=0.5)
            contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            for cnt in contours:
                ax.plot(cnt[:, 0, 0], cnt[:, 0, 1], color='orange', linewidth=2)

        ax.set_title(f"RF Frame {i}")
    ax.axis('off')

plt.suptitle("SAM 2 Tracking on Roboflow Dataset Frames", fontsize=14)
plt.tight_layout()
plt.show()



> Using Roboflow Image Dataset



In [ ]:
API_KEY = 'YOUR_API_KEY_HERE'
WORKSPACE = 'YOUR_WORKSPACE_HERE'
PROJECT_NAME = 'YOUR_PROJECT_NAME_HERE'
VERSION = 1

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
dataset = project.version(ROBOFLOW_VERSION).download('png-mask-semantic')
RF_IMG_DIR = dataset.location

In [ ]:
all_image_paths = []
for split in ['train', 'valid', 'test']:
  split_dir = os.path.join(RF_IMG_DIR, split)
  if os.path.isdir(split_dir):
    imgs = [
        os.path.join(split_dir, f)
        for f in os.listdir(split_dir)
        if f.endswith(('.jpg', '.jpeg', '.png'))
    ]
    all_image_paths.extend(imgs)
    print(f"   {split:6s}: {len(imgs)} images")

print(f"\n   Total: {len(all_image_paths)} images")

In [ ]:
def sam_segment_image(predictor, image_path, strategy='center'):
  img = cv2.imread(image_path)
  if img is None:
    raise ValueError(f"Cannot read image: {image_path}")
  img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
  img_h, img_w = img.shape[:2]

  with torch.inference_mode():
    predictor.set_image(img)

    if strategy == 'center':
      masks_out, scores_out, _ = predictor.predict(
          point_coords    = np.array([[img_w//2, img_h//2]]),
          point_labels    = np.array([1]),
          multimask_output = True
      )
      best_mask = masks_out[int(np.argmax(scores_out))]
      best_score = scores_out.max()

    elif strategy == 'box':
      margin = max(5, min(img_w, img_h) // 20)
      masks_out, scores_out, _ = predictor.predict(
          box              = np.array([margin, margin, img_w - margin, img_h - margin]),
          multimask_output = False
      )
      best_mask = masks_out[0]
      best_score = scores_out[0]

    else:
      raise ValueError(f"Unknown strategy: {strategy!r}. Use 'center' or 'box'.")

  return img, best_mask, float(best_score)

In [ ]:
BATCH_SIZE = min(6, len(all_image_paths))
STRATEGY = 'center'

results = []

for img_path in all_image_paths[:BATCH_SIZE]:
  try:
    img_rgb, mask, score = sam_segment_image(predictor, img_path, strategy=STRATEGY)
    results.append({'path': img_path, 'image': img_rgb, 'mask': mask, 'score': score})
  except Exception as e:
    print(f"   {os.path.basename(img_path)}: {e}")

In [ ]:
n = len(results)
if n == 0:
    print("No results to display.")
else:
    fig, axes = plt.subplots(3, n, figsize=(4 * n, 12))
    if n == 1:
        axes = axes.reshape(3, 1)
    for col, res in enumerate(results):
        img_rgb = res['image']
        mask    = res['mask'] > 0.0
        h, w    = img_rgb.shape[:2]
        fname   = os.path.basename(res['path'])

        axes[0, col].imshow(img_rgb)
        axes[0, col].set_title(fname[:20], fontsize=8)
        axes[0, col].axis('off')

        axes[1, col].imshow(mask, cmap='gray')
        axes[1, col].set_title(f'Mask (score={res["score"]:.2f})', fontsize=8)
        axes[1, col].axis('off')

        overlay = img_rgb.astype(float).copy()
        overlay[mask] = overlay[mask] * 0.4 + np.array([0, 160, 100]) * 0.6
        axes[2, col].imshow(overlay.clip(0, 255).astype(np.uint8))
        axes[2, col].set_title(f'{mask.mean()*100:.1f}% coverage', fontsize=8)
        axes[2, col].axis('off')

    axes[0, 0].set_ylabel('Original',     fontsize=10, rotation=90, labelpad=5)
    axes[1, 0].set_ylabel('SAM Mask',     fontsize=10, rotation=90, labelpad=5)
    axes[2, 0].set_ylabel('Overlay',      fontsize=10, rotation=90, labelpad=5)

    plt.suptitle(
        f'SAM Zero-Shot Batch Inference on Roboflow Dataset ({n} images, strategy={STRATEGY!r})',
        fontsize=12
    )
    plt.tight_layout()
    plt.show()

    avg_score    = np.mean([r['score'] for r in results])
    avg_coverage = np.mean([r['mask'].mean() for r in results]) * 100
    print(f" Batch stats: avg score={avg_score:.3f}  avg coverage={avg_coverage:.1f}%")
    print(f" To process all {len(all_image_paths)} images change BATCH_SIZE above")

```
When to fine-tune:
  Your domain differs a lot from natural images (medical, satellite, microscopy)
  Zero-shot SAM misses fine details specific to your domain
  You need consistent quality on a specific object type

Parameter-efficient strategy:
  Freeze: Image Encoder  (largest — keep its strong general features)
  Train:  Mask Decoder + Prompt Encoder  (small output layers — learn your domain)

Full fine-tuning loop needs:
 1. Dataset with images + ground-truth binary masks
 2. Prompt strategy (center points, GT boxes, or automatic prompts)
 3. Loss = Dice + BCE on output masks (same pattern as U-Net training)

Only ~10–20% of parameters need training → fast + low overfitting
```

In [ ]:
for param in sam2_model.image_encoder.parameters():
  param.requires_grad = False

for param in sam2_model.sam_mask_decoder.parameters():
  param.requires_grad = True

for param in sam2_model.sam_prompt_encoder.parameters():
  param.requires_grad = True

total_params = sum(p.numel() for p in sam2_model.parameters())
trainable_params = sum(p.numel() for p in sam2_model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(" Fine-tuning parameter setup:")
print(f"   Total params     : {total_params/1e6:.1f}M")
print(f"   Frozen params    : {frozen_params/1e6:.1f}M  (Image Encoder — frozen)")
print(f"   Trainable params : {trainable_params/1e6:.1f}M  ({trainable_params/total_params*100:.1f}% of total)")